# KAMP Press Equipment Anomaly Detection

이 notebook은 원래 `KMAP.ipynb` 실험을 GitHub 재현용으로 정리한 버전입니다. 재사용 가능한 전처리·feature engineering·평가 로직은 `src/`로 분리했습니다.


In [ ]:
from pathlib import Path
from src.config import GAP_THRESHOLD_SECONDS, WINDOW_SIZE
from src.data import load_raw_data, create_segments
from src.features import build_feature_dataset
from src.evaluate import evaluate_repeated_group_cv, summarize


## 1. Load raw data


In [ ]:
DATA_DIR = Path('../data')
normal, abnormal = load_raw_data(DATA_DIR)
print('normal:', normal.shape)
print('abnormal:', abnormal.shape)


## 2. Time-aware segmentation

인접 `TimeStamp` 간격이 0.11초를 초과하면 새로운 segment로 정의합니다.


In [ ]:
normal_seg = create_segments(normal, GAP_THRESHOLD_SECONDS)
abnormal_seg = create_segments(abnormal, GAP_THRESHOLD_SECONDS)
print('normal segments:', normal_seg['segment_id'].nunique())
print('abnormal segments:', abnormal_seg['segment_id'].nunique())


## 3. 1-second non-overlapping windows

10 Hz sampling을 기준으로 10 sample ≈ 1초 window를 생성하고, 각 센서별 mean/std/RMS/min/max를 추출합니다.


In [ ]:
feature_df = build_feature_dataset(normal_seg, abnormal_seg, WINDOW_SIZE)
feature_df.head()


## 4. Repeated group-aware validation

`StratifiedGroupKFold`를 사용해 동일 segment의 window가 train/test에 동시에 들어가는 것을 방지합니다.


In [ ]:
window_metrics, segment_metrics = evaluate_repeated_group_cv(feature_df)
display(summarize(window_metrics))
display(summarize(segment_metrics))


## Notes

- 본 저장소의 대표 성능은 개발 단계 repeated Group CV 결과입니다.
- usable abnormal segment 수가 적어 segment-level 점수는 보조적으로 해석합니다.
- 상세 오류분석(Segment 16/20), SHAP, 모델 비교 및 tuning 의사결정은 README와 `docs/`에 정리했습니다.
